In [54]:
!pip install rank-bm25

In [55]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/sample_submission.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/dataset-metadata.json
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/baseline_submission.csv


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
base = "/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/"

documents = pd.read_csv(base + "documents.csv")
train_queries = pd.read_csv(base + "train_queries.csv")
qrels_train = pd.read_csv(base + "qrels_train.csv")
test_queries = pd.read_csv(base + "test_queries.csv")

print(documents.columns.tolist())
documents.head()

['document_id', 'title', 'text', 'source', 'crop', 'country', 'origin', 'source_url', 'license']


,document_id,title,text,source,crop,country,origin,source_url,license
0,1,Drought and erratic rainfall: the risk to crops,Drought and erratic rainfall and its impact on...,FAO,(general),Kenya,synthetic,NaN,synthetic (CC0)
1,2,Adapting to drought and erratic rainfall (Guin...,Adapting to drought and erratic rainfall in th...,IITA,(general),Tanzania,synthetic,NaN,synthetic (CC0)
2,3,Adapting to drought and erratic rainfall (High...,Adapting to drought and erratic rainfall in th...,ICRISAT,(general),Nigeria,synthetic,NaN,synthetic (CC0)
3,4,Adapting to drought and erratic rainfall (Humi...,Adapting to drought and erratic rainfall in th...,FAO,(general),Mali,synthetic,NaN,synthetic (CC0)
4,5,Adapting to drought and erratic rainfall (Sahel),Adapting to drought and erratic rainfall in th...,FAO,(general),Tanzania,synthetic,NaN,synthetic (CC0)


In [58]:
docs = pd.read_csv(base + "documents.csv")
test = pd.read_csv(base + "test_queries.csv")

docs["title"] = docs["title"].fillna("")
docs["text"] = docs["text"].fillna("")

# Build TF-IDF index over title + text
vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
doc_mat = vec.fit_transform(docs["title"] + ". " + docs["text"])

# Retrieve top-5 for each query -> LONG format (QueryId, DocumentId); row order = rank
q_mat = vec.transform(test["query"])
sims = cosine_similarity(q_mat, doc_mat)

rows = []
for i, qid in enumerate(test["query_id"]):
    top5 = sims[i].argsort()[::-1][:5]
    for rank in top5:
        rows.append({"QueryId": qid, "DocumentId": int(docs.iloc[rank]["document_id"])})

pd.DataFrame(rows).to_csv("submission.csv", index=False)
print("Wrote submission.csv")

Wrote submission.csv


In [59]:
# Loading ground truth relevant docs
qrels = pd.read_csv(base + "qrels_train.csv")
train_q = pd.read_csv(base + "train_queries.csv")

# Matching train queries with docs using TF-IDF
q_mat_train = vec.transform(train_q["query"])
sims_train = cosine_similarity(q_mat_train, doc_mat)

# Calculating recall / MRR
hits = 0
reciprocal_ranks = []

for i, row in train_q.iterrows():
    qid = row["query_id"]
    true_doc = qrels[qrels["query_id"] == qid]["document_id"].values
    
    top5_idx = sims_train[i].argsort()[::-1][:5]
    pred_docs = docs.iloc[top5_idx]["document_id"].values
    
    # Checking if correct document is in top 5
    found = [rank + 1 for rank, d in enumerate(pred_docs) if d in true_doc]
    if found:
        hits += 1
        reciprocal_ranks.append(1.0 / found[0])
    else:
        reciprocal_ranks.append(0.0)

print(f"Recall@5: {hits / len(train_q):.4f}")
print(f"MRR@5:    {np.mean(reciprocal_ranks):.4f}")

Recall@5: 0.9773
MRR@5:    0.9690


In [60]:
from rank_bm25 import BM25Okapi

# Preparing documents for BM25
docs["title"] = docs["title"].fillna("")
docs["text"] = docs["text"].fillna("")

# Combining title and text
docs["content"] = docs["title"] + ". " + docs["text"]

# Tokenizing the documents
doc_tokens = docs["content"].str.lower().str.split().tolist()

# Building the BM25 index
bm25 = BM25Okapi(
    doc_tokens,
    k1=0.82,
    b=0.68
)

In [61]:
import re
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [62]:
def tokenize(text):
    # Extracts words and removes numbers, punctuation, and common stop words
    tokens = re.findall(r'\b[a-zA-Z]{2,}\b', str(text).lower())
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS]

In [63]:
# Load and clean document fields
docs = pd.read_csv(base + "documents.csv")
docs["title"] = docs["title"].fillna("")
docs["text"] = docs["text"].fillna("")

# Combine title and text content
docs["content"] = docs["title"] + ". " + docs["text"]

# Tokenize corpus for BM25
doc_tokens = [tokenize(doc) for doc in docs["content"]]

# Build BM25 index with standard parameters
bm25 = BM25Okapi(doc_tokens, k1=1.5, b=0.75)

In [64]:
qrels = pd.read_csv(base + "qrels_train.csv")
train_q = pd.read_csv(base + "train_queries.csv")

hits = 0
reciprocal_ranks = []

for i, row in train_q.iterrows():
    qid = row["query_id"]
    true_doc = qrels[qrels["query_id"] == qid]["document_id"].values
    
    # Tokenize query using the same preprocess function
    query_tokens = tokenize(row["query"])
    
    sims = bm25.get_scores(query_tokens)
    top5_idx = sims.argsort()[::-1][:5]
    pred_docs = docs.iloc[top5_idx]["document_id"].values
    
    found = [rank + 1 for rank, d in enumerate(pred_docs) if d in true_doc]
    if found:
        hits += 1
        reciprocal_ranks.append(1.0 / found[0])
    else:
        reciprocal_ranks.append(0.0)

print(f"Recall@5: {hits / len(train_q):.4f}")
print(f"MRR@5:    {np.mean(reciprocal_ranks):.4f}")

Recall@5: 0.9708
MRR@5:    0.8720


In [65]:
# Generate predictions on the test queries using BM25
test = pd.read_csv(base + "test_queries.csv")

rows = []
for i, row in test.iterrows():
    qid = row["query_id"]
    query_tokens = tokenize(row["query"])
    sims = bm25.get_scores(query_tokens)
    top5_idx = sims.argsort()[::-1][:5]
    
    for rank_idx in top5_idx:
        rows.append({
            "QueryId": qid,
            "DocumentId": int(docs.iloc[rank_idx]["document_id"])
        })

# Save output to submission.csv
submission_df = pd.DataFrame(rows)
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")
print(submission_df.head())

Saved submission.csv successfully!
   QueryId  DocumentId
0     1001           1
1     1001           5
2     1001           2
3     1001           3
4     1001           4
